# CloudCompute: robust fine-tuning
Отдельная ветка моделей на усиленных геометрических и фотометрических аугментациях.

In [ ]:
REPO_URL = "https://github.com/frest1ler/text-orientation-classification.git"
BRANCH = "main"
QUICK_RUN = True
AUGMENTATION_PROFILE = "robust"  # standard | robust
EPOCHS = 5
LEARNING_RATE = 2e-5
BATCH_SIZE = 64
VALIDATION_BATCH_SIZE = 128
NUM_WORKERS = 2
RESUME_TRAINING = True
PROMOTE_CHAMPION = False
RUN_TESTS = False
REPO_DIR = "/root/text-orientation-classification"
STATE_DIR = "/root/text-orientation-state"

In [ ]:
import os, subprocess, sys
from pathlib import Path
repo = Path(REPO_DIR)
if not (repo / ".git").is_dir(): subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if RUN_TESTS: subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
from src.registry import select_champion
bundle = select_champion(Path(STATE_DIR) / "registry", "mobilenet_v3_large")
print("Initial checkpoint:", bundle.checkpoint_path)

In [ ]:
mode = "quick" if QUICK_RUN else "full"
run_name = f"mobilenet_{AUGMENTATION_PROFILE}_{mode}"
run_dir = Path("artifacts/experiments") / run_name
recovery = Path(STATE_DIR) / "training/recovery/robust/mobilenet_v3_large" / mode
command = [sys.executable, "-m", "scripts.train_robust", "--output-dir", str(run_dir), "--recovery-dir", str(recovery), "--augmentation-profile", AUGMENTATION_PROFILE, "--initial-checkpoint", str(bundle.checkpoint_path), "--epochs", str(EPOCHS), "--learning-rate", str(LEARNING_RATE), "--batch-size", str(BATCH_SIZE), "--validation-batch-size", str(VALIDATION_BATCH_SIZE), "--num-workers", str(NUM_WORKERS)]
if QUICK_RUN: command += ["--train-base-samples", "2048", "--validation-base-samples", "512"]
if RESUME_TRAINING: command.append("--resume")
subprocess.run(command, check=True)
if PROMOTE_CHAMPION and not QUICK_RUN:
    subprocess.run([sys.executable, "-m", "scripts.promote_robust_champion", "--run-dir", str(run_dir), "--project-dir", STATE_DIR], check=True)
exports = Path(STATE_DIR) / "training/runs/robust/mobilenet_v3_large" / mode
exports.mkdir(parents=True, exist_ok=True)
archive = exports / run_name
subprocess.run(["zip", "-qr", str(archive) + ".zip", str(run_dir)], check=True)
print("Result:", str(archive) + ".zip")